# Sumativa 4 — Taller 3
## Integración, limpieza y análisis de ventas con Apache Spark

**Curso:** Gestión de Datos y Tecnologías Big Data — MCDI502  
**Grupo:** 7  
**Entorno:** Databricks

### Objetivo

Integrar las ventas de octubre–diciembre de 2024 de las sucursales de Viña del Mar, Santiago y Concepción. El flujo revisa la calidad de los datos, aplica las reglas de limpieza solicitadas, crea variables para el análisis y ejecuta consultas con Spark SQL.

El notebook está organizado en las cinco evidencias principales de la rúbrica: carga e integración, limpieza, transformación, validación de calidad y consultas SQL.

# PARTE 1: Configuración inicial y carga de archivos

Carga los tres CSV en el almacenamiento de tu espacio Databricks y ajusta las rutas de esta celda. Se conserva `dbfs:/FileStore/tables/` como ruta del taller; si tu espacio utiliza Volumes, escribe la ruta `/Volumes/...` correspondiente.

Databricks proporciona `spark` y `dbutils`. El flujo utiliza conversiones tolerantes para que un formato de fecha incompatible no interrumpa los demás intentos, incluso con modo ANSI activado.

In [0]:
from pyspark.sql import functions as F

print("Versión de Spark:", spark.version)

# Configuración de rutas en Unity Catalog
CATALOGO = "workspace"
ESQUEMA = "default"
VOLUMEN = "datos_ventas"

BASE = f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUMEN}"

RUTA_VINA = f"{BASE}/ventas_vina_del_mar.csv"
RUTA_SANTIAGO = f"{BASE}/ventas_santiago.csv"
RUTA_CONCEPCION = f"{BASE}/ventas_concepcion.csv"

rutas = [RUTA_VINA, RUTA_SANTIAGO, RUTA_CONCEPCION]
for ruta in rutas:
    try:
        dbutils.fs.ls(ruta)
        print("Archivo disponible:", ruta)
    except Exception as error:
        raise FileNotFoundError(
            f"No se encontró {ruta}. Carga el CSV en DBFS y vuelve a ejecutar esta celda."
        ) from error

Versión de Spark: 4.2.0
Archivo disponible: /Volumes/workspace/default/datos_ventas/ventas_vina_del_mar.csv
Archivo disponible: /Volumes/workspace/default/datos_ventas/ventas_santiago.csv
Archivo disponible: /Volumes/workspace/default/datos_ventas/ventas_concepcion.csv


## Carga de los tres DataFrames

Se leen inicialmente como texto para evitar que los formatos mezclados de fecha y precio se conviertan de manera incorrecta. Las conversiones se realizan de forma explícita en la etapa de limpieza.

In [0]:
opciones_csv = {"header": True, "inferSchema": False}

df_vina = spark.read.options(**opciones_csv).csv(RUTA_VINA)
df_santiago = spark.read.options(**opciones_csv).csv(RUTA_SANTIAGO)
df_concepcion = spark.read.options(**opciones_csv).csv(RUTA_CONCEPCION)

def inspeccionar(nombre, df):
    print(f"\n--- {nombre} ---")
    print("Filas:", df.count())
    df.printSchema()
    df.show(5, truncate=False)

inspeccionar("Viña del Mar", df_vina)
inspeccionar("Santiago", df_santiago)
inspeccionar("Concepción", df_concepcion)


--- Viña del Mar ---
Filas: 105
root
 |-- id_transaccion: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- id_producto: string (nullable = true)
 |-- producto_nombre: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: string (nullable = true)
 |-- precio_unitario: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- sucursal: string (nullable = true)

+--------------+----------+-----------+-----------------+--------------+--------+---------------+----------+------------+
|id_transaccion|fecha     |id_producto|producto_nombre  |categoria     |cantidad|precio_unitario|id_cliente|sucursal    |
+--------------+----------+-----------+-----------------+--------------+--------+---------------+----------+------------+
|TRX10030      |01/10/2024|PROD889    |Mouse Inalámbrico|Almacenamiento|8       |79990          |CLI1158   |Viña del Mar|
|TRX10065      |2024/10/13|PROD403    |Teclado Mecánico |Almacenamiento|-1      |1        

## Unión de los DataFrames

Los tres archivos tienen las mismas columnas. unionByName conserva la correspondencia por nombre y permite comprobar que ninguna fila se pierda durante la integración.

In [0]:
filas_vina = df_vina.count()
filas_santiago = df_santiago.count()
filas_concepcion = df_concepcion.count()

df_ventas_unificado = (
    df_vina
    .unionByName(df_santiago)
    .unionByName(df_concepcion)
)

filas_esperadas = filas_vina + filas_santiago + filas_concepcion
filas_unificadas = df_ventas_unificado.count()

print("Filas Viña del Mar:", filas_vina)
print("Filas Santiago:", filas_santiago)
print("Filas Concepción:", filas_concepcion)
print("Suma de filas originales:", filas_esperadas)
print("Filas del DataFrame unificado:", filas_unificadas)

if filas_unificadas != filas_esperadas:
    raise ValueError("La unión no conserva la suma de filas de las tres fuentes.")

print("Validación de unión: correcta")

Filas Viña del Mar: 105
Filas Santiago: 125
Filas Concepción: 85
Suma de filas originales: 315
Filas del DataFrame unificado: 315
Validación de unión: correcta


## Análisis de la integración de datos

La integración de las tres fuentes permitió consolidar **315 registros de ventas**, distribuidos entre Santiago (125), Viña del Mar (105) y Concepción (85), equivalentes al 39,68%, 33,33% y 26,98% del total, respectivamente.

La validación realizada confirma que el número de registros del DataFrame unificado coincide exactamente con la suma de los registros originales. Esto permite descartar pérdidas de filas durante la operación `unionByName()`.

Sin embargo, esta comprobación garantiza únicamente la conservación de los registros durante la unión, no su calidad. Por lo tanto, resulta necesario evaluar posteriormente la existencia de duplicados, valores faltantes e inconsistencias que puedan afectar la confiabilidad del análisis comercial.

# PARTE 2: Calidad inicial antes de limpiar

Se consideran como faltantes tanto los valores nulos como las cadenas vacías. Esto es necesario porque en estos CSV algunas categorías aparecen vacías, aunque no estén representadas como NULL.

In [0]:
columnas_originales = df_ventas_unificado.columns

def contar_faltantes(df, columnas):
    expresiones = []
    for columna in columnas:
        valor_texto = F.col(columna).cast("string")
        expresiones.append(
            F.sum(
                F.when(
                    F.col(columna).isNull() | (F.trim(valor_texto) == ""),
                    1
                ).otherwise(0)
            ).alias(columna)
        )
    return df.select(expresiones).first().asDict()

filas_antes = df_ventas_unificado.count()
duplicados_antes = filas_antes - df_ventas_unificado.dropDuplicates().count()
nulos_antes = contar_faltantes(df_ventas_unificado, columnas_originales)

cantidad_num_antes = F.expr("try_cast(cantidad AS INT)")
cantidad_invalida_antes = df_ventas_unificado.filter(
    (F.trim(F.col("cantidad")) != "") & cantidad_num_antes.isNull()
).count()
cantidad_no_positiva_antes = df_ventas_unificado.filter(cantidad_num_antes <= 0).count()

print("Filas iniciales:", filas_antes)
print("Duplicados completos:", duplicados_antes)
print("Filas con cantidad no convertible:", cantidad_invalida_antes)
print("Filas con cantidad menor o igual a cero:", cantidad_no_positiva_antes)

tabla_nulos_antes = spark.createDataFrame(
    [(columna, int(nulos_antes[columna])) for columna in columnas_originales],
    ["columna", "faltantes_antes"]
)
tabla_nulos_antes.show(truncate=False)

Filas iniciales: 315
Duplicados completos: 15
Filas con cantidad no convertible: 0
Filas con cantidad menor o igual a cero: 27
+---------------+---------------+
|columna        |faltantes_antes|
+---------------+---------------+
|id_transaccion |0              |
|fecha          |0              |
|id_producto    |0              |
|producto_nombre|0              |
|categoria      |46             |
|cantidad       |0              |
|precio_unitario|0              |
|id_cliente     |0              |
|sucursal       |0              |
+---------------+---------------+



## Diagnóstico inicial de calidad de los datos

El análisis exploratorio evidencia tres problemas principales de calidad:

- **Registros duplicados:** se identifican 15 filas completamente duplicadas, equivalentes al 4,76% de los registros originales. Su permanencia podría sobreestimar el número de ventas y los ingresos.
- **Categorías faltantes:** se observan 46 registros sin información de categoría (14,60%). Esta situación dificulta la clasificación de productos y la comparación del desempeño por línea comercial.
- **Cantidades no positivas:** se detectan 27 registros (8,57%) con cantidades menores o iguales a cero. Dado que el objetivo es analizar ventas, estos registros requieren depuración para evitar distorsiones en los montos calculados.

Estos problemas justifican la implementación de un proceso de limpieza previo al análisis. Es importante señalar que los porcentajes corresponden a problemas que pueden superponerse en un mismo registro y, por lo tanto, no deben sumarse para estimar la proporción total de datos afectados.

# PARTE 3: Limpieza y depuración

Las decisiones se aplican de forma visible:

- Las filas completamente duplicadas se eliminan con dropDuplicates().
- Las categorías vacías se imputan como Sin categoría para conservar esas ventas y diferenciarlas de una categoría real.
- Los nombres de producto se llevan a minúsculas y se eliminan espacios sobrantes.
- Las fechas se convierten de manera tolerante usando los formatos observados: dd/MM/yyyy, dd-MM-yyyy, yyyy/MM/dd y yyyy-MM-dd.
- Los precios observados corresponden a pesos enteros, con símbolo $ opcional y espacios. Se retiran esos símbolos antes de convertir a número.

In [0]:
# 1. Duplicados completos: conservar una sola copia de cada fila original.
df_limpio = df_ventas_unificado.dropDuplicates()
filas_sin_duplicados = df_limpio.count()
print("Duplicados eliminados:", filas_antes - filas_sin_duplicados)
print("Filas sin duplicados:", filas_sin_duplicados)

# 2. Imputar categorías faltantes para conservar sus ventas.
df_limpio = df_limpio.withColumn(
    "categoria",
    F.when(F.trim(F.col("categoria")) == "", F.lit(None))
     .otherwise(F.trim(F.col("categoria")))
).fillna({"categoria": "Sin categoría"})

# 3. Normalizar espacios y mayúsculas de los nombres.
df_limpio = df_limpio.withColumn(
    "producto_nombre",
    F.lower(F.trim(F.regexp_replace(F.col("producto_nombre"), r"\s+", " ")))
)

# 4. Definir una sola vez las conversiones para limpieza y controles.
formatos_fecha = ["dd/MM/yyyy", "dd-MM-yyyy", "yyyy/MM/dd", "yyyy-MM-dd"]
fecha_convertida = F.coalesce(*[
    F.expr(f"try_to_timestamp(trim(fecha), '{formato}')").cast("date")
    for formato in formatos_fecha
])
precio_convertido = F.expr(
    r"try_cast(regexp_replace(trim(precio_unitario), r'[$\s]', '') AS DOUBLE)"
)
cantidad_convertida = F.expr("try_cast(trim(cantidad) AS INT)")

# Estos controles se calculan sobre todas las filas originales.
fechas_invalidas_antes = df_ventas_unificado.filter(fecha_convertida.isNull()).count()
precios_invalidos_antes = df_ventas_unificado.filter(precio_convertido.isNull()).count()
totales_negativos_antes = df_ventas_unificado.filter(
    cantidad_convertida * precio_convertido < 0
).count()

# Conservar el valor recibido para poder revisar una fila rechazada.
df_limpio = (
    df_limpio
    .withColumn("_fecha_original", F.col("fecha"))
    .withColumn("_precio_original", F.col("precio_unitario"))
    .withColumn("_cantidad_original", F.col("cantidad"))
    .withColumn("fecha", fecha_convertida)
    .withColumn("precio_unitario", precio_convertido)
    .withColumn("cantidad", cantidad_convertida)
)
print("Fechas inválidas o nulas en origen:", fechas_invalidas_antes)
print("Precios inválidos o nulos en origen:", precios_invalidos_antes)

Duplicados eliminados: 15
Filas sin duplicados: 300
Fechas inválidas o nulas en origen: 0
Precios inválidos o nulos en origen: 0


In [0]:
print("Esquema después de la limpieza:")
df_limpio.select(*columnas_originales).printSchema()
df_limpio.select(*columnas_originales).show(5, truncate=False)

Esquema después de la limpieza:
root
 |-- id_transaccion: string (nullable = true)
 |-- fecha: date (nullable = true)
 |-- id_producto: string (nullable = true)
 |-- producto_nombre: string (nullable = true)
 |-- categoria: string (nullable = false)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- sucursal: string (nullable = true)

+--------------+----------+-----------+---------------+-------------+--------+---------------+----------+------------+
|id_transaccion|fecha     |id_producto|producto_nombre|categoria    |cantidad|precio_unitario|id_cliente|sucursal    |
+--------------+----------+-----------+---------------+-------------+--------+---------------+----------+------------+
|TRX10053      |2024-11-05|PROD358    |impresora laser|Sin categoría|0       |349990.0       |CLI1659   |Viña del Mar|
|TRX10012      |2024-12-20|PROD961    |disco ssd 1tb  |Periféricos  |3       |79990.0        |CLI1009  

# PARTE 4: Transformación y validación

Se excluyen cantidades nulas o no positivas, fechas inválidas, precios inválidos y precios totales negativos. Cada fila rechazada se conserva en `df_rechazadas`, junto con los valores originales y un motivo. Si incumple varias reglas, se registra el primer motivo para no contar dos veces la misma fila.

Se comprueba que las filas sin duplicados sean exactamente la suma de las aceptadas y las rechazadas.

In [0]:
df_transformado = (
    df_limpio
    .withColumn("precio_total", F.col("cantidad") * F.col("precio_unitario"))
    .withColumn("dia_semana", F.date_format(F.col("fecha"), "EEEE"))
    .withColumn(
        "motivo_rechazo",
        F.when(F.col("cantidad").isNull(), "Cantidad inválida o nula")
         .when(F.col("cantidad") <= 0, "Cantidad menor o igual a cero")
         .when(F.col("fecha").isNull(), "Fecha inválida o nula")
         .when(F.col("precio_unitario").isNull(), "Precio inválido o nulo")
         .when(F.col("precio_total") < 0, "Precio total negativo")
    )
)
df_rechazadas = df_transformado.filter(F.col("motivo_rechazo").isNotNull())
df_ventas_final = (
    df_transformado.filter(F.col("motivo_rechazo").isNull())
    .drop("motivo_rechazo", "_fecha_original", "_precio_original", "_cantidad_original")
)

filas_rechazadas = df_rechazadas.count()
filas_validas = df_ventas_final.count()
print("Filas sin duplicados:", filas_sin_duplicados)
print("Filas rechazadas:", filas_rechazadas)
print("Filas aceptadas:", filas_validas)
assert filas_sin_duplicados == filas_rechazadas + filas_validas
df_rechazadas.groupBy("motivo_rechazo").count().show(truncate=False)

negativos_finales = df_ventas_final.filter(F.col("precio_total") < 0).count()
print("Precios totales negativos restantes:", negativos_finales)
assert negativos_finales == 0

Filas sin duplicados: 300
Filas rechazadas: 26
Filas aceptadas: 274
+-----------------------------+-----+
|motivo_rechazo               |count|
+-----------------------------+-----+
|Cantidad menor o igual a cero|26   |
+-----------------------------+-----+

Precios totales negativos restantes: 0


## Evaluación del proceso de depuración

El proceso de limpieza redujo el conjunto de datos desde 315 registros originales hasta **274 registros válidos**, conservando el 86,98% de las observaciones iniciales.

La reducción se explica por la eliminación de 15 registros completamente duplicados y el rechazo de 26 registros con cantidades menores o iguales a cero. La diferencia entre las 27 cantidades no positivas identificadas inicialmente y los 26 rechazos posteriores se debe a que uno de esos registros también estaba duplicado.

La estrategia aplicada permite distinguir dos tratamientos:

1. **Corrección y conservación:** los registros con categoría faltante se mantienen mediante la etiqueta `Sin categoría`, evitando perder información de ventas y sin atribuirles una clasificación comercial desconocida.
2. **Exclusión justificada:** los registros con cantidades no positivas se separan del conjunto analítico, conservando sus antecedentes en `df_rechazadas` para asegurar trazabilidad.

Finalmente, la validación confirma que no permanecen precios totales negativos y que la suma de registros aceptados y rechazados coincide con los registros posteriores a la deduplicación.

Por tanto, el procedimiento mejora la consistencia del conjunto de datos y permite construir una base apta para los análisis solicitados, aunque todavía requiere examinar posibles anomalías en los precios.

In [0]:
print("Cantidad de sucursales únicas:", df_ventas_final.select("sucursal").distinct().count())
df_ventas_final.select("sucursal").distinct().orderBy("sucursal").show(truncate=False)

print("Cantidad de categorías únicas:", df_ventas_final.select("categoria").distinct().count())
df_ventas_final.select("categoria").distinct().orderBy("categoria").show(truncate=False)

print("Ejemplo de columnas calculadas:")
df_ventas_final.select(
    "id_transaccion", "fecha", "dia_semana", "cantidad",
    "precio_unitario", "precio_total"
).show(5, truncate=False)

Cantidad de sucursales únicas: 3
+------------+
|sucursal    |
+------------+
|Concepción  |
|Santiago    |
|Viña del Mar|
+------------+

Cantidad de categorías únicas: 6
+--------------+
|categoria     |
+--------------+
|Almacenamiento|
|Audio         |
|Computación   |
|Periféricos   |
|Redes         |
|Sin categoría |
+--------------+

Ejemplo de columnas calculadas:
+--------------+----------+----------+--------+---------------+------------+
|id_transaccion|fecha     |dia_semana|cantidad|precio_unitario|precio_total|
+--------------+----------+----------+--------+---------------+------------+
|TRX10012      |2024-12-20|Friday    |3       |79990.0        |239970.0    |
|TRX10035      |2024-11-09|Saturday  |2       |199990.0       |399980.0    |
|TRX10017      |2024-10-05|Saturday  |3       |449990.0       |1349970.0   |
|TRX10090      |2024-12-19|Thursday  |5       |159990.0       |799950.0    |
|TRX10059      |2024-11-29|Friday    |2       |199990.0       |399980.0    |
+--------

## Detección de valores atípicos con IQR

Se calcula el rango intercuartílico de precio_unitario. Los registros fuera de Q1 − 1,5 × IQR o Q3 + 1,5 × IQR se marcan con es_outlier_precio. Se conservan en el DataFrame para revisión; no se eliminan automáticamente. El IQR se calcula globalmente: un precio fuera del umbral requiere revisión, pero no demuestra por sí solo un error. Los límites también pueden dejar sin marcar precios muy bajos; se revisan al interpretar mínimos y máximos.

In [0]:
if filas_validas == 0:
    raise ValueError("No quedan ventas válidas para calcular los cuartiles.")

q1, q3 = df_ventas_final.approxQuantile(
    "precio_unitario", [0.25, 0.75], 0.0
)
iqr = q3 - q1
limite_inferior = q1 - (1.5 * iqr)
limite_superior = q3 + (1.5 * iqr)

df_ventas_final = df_ventas_final.withColumn(
    "es_outlier_precio",
    (F.col("precio_unitario") < F.lit(limite_inferior))
    | (F.col("precio_unitario") > F.lit(limite_superior))
)
df_outliers = df_ventas_final.filter(F.col("es_outlier_precio"))

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Límite inferior:", limite_inferior)
print("Límite superior:", limite_superior)
print("Outliers detectados:", df_outliers.count())
df_outliers.select(
    "id_transaccion", "sucursal", "producto_nombre", "precio_unitario",
    "precio_total", "es_outlier_precio"
).show(20, truncate=False)

# Medir cuánto influyen los registros marcados en el total vendido.
ventas_totales = df_ventas_final.agg(F.sum("precio_total")).first()[0]
ventas_outliers = df_outliers.agg(F.sum("precio_total")).first()[0] or 0.0
porcentaje_outliers = 100 * ventas_outliers / ventas_totales if ventas_totales else None
print("Monto total vendido:", ventas_totales)
print("Monto correspondiente a outliers:", ventas_outliers)
print("Porcentaje del total en outliers:", round(porcentaje_outliers, 2) if porcentaje_outliers is not None else "No aplica")


Q1: 79990.0
Q3: 299990.0
IQR: 220000.0
Límite inferior: -250010.0
Límite superior: 629990.0
Outliers detectados: 7
+--------------+------------+---------------------+---------------+------------+-----------------+
|id_transaccion|sucursal    |producto_nombre      |precio_unitario|precio_total|es_outlier_precio|
+--------------+------------+---------------------+---------------+------------+-----------------+
|TRX10037      |Viña del Mar|webcam hd            |8888888.0      |3.5555552E7 |true             |
|TRX20037      |Santiago    |mouse inalambrico    |9999990.0      |2.999997E7  |true             |
|TRX20047      |Santiago    |router wifi          |8888888.0      |7.9999992E7 |true             |
|TRX20119      |Santiago    |auriculares bluetooth|9999990.0      |6.999993E7  |true             |
|TRX20001      |Santiago    |disco ssd 1tb        |9999990.0      |6.999993E7  |true             |
|TRX30045      |Concepción  |laptop gamer         |9999990.0      |7.999992E7  |true         

## Interpretación de los valores atípicos y su impacto

La aplicación del método del rango intercuartílico (IQR) permitió identificar **7 registros con precios unitarios atípicos**, equivalentes al 2,55% de las 274 ventas válidas.

Los cuartiles obtenidos fueron Q1 = $79.990 y Q3 = $299.990, generando un IQR de $220.000. Al aplicar el criterio de 1,5 veces el IQR, se estableció un límite superior de $629.990, sobre el cual se identificaron los siete registros.

El principal hallazgo es que estos registros, pese a representar una proporción reducida del conjunto de datos, acumulan **$385.555.274**, equivalentes al **60,33% de los $639.025.343** registrados como ventas.

Esta concentración evidencia una elevada sensibilidad del monto total a un pequeño grupo de observaciones extremas. Por consiguiente, los resultados agregados podrían ofrecer una representación distorsionada del desempeño comercial habitual si estos valores corresponden a errores de registro.

Se decidió **conservar los registros atípicos, pero identificarlos explícitamente**, porque un valor estadísticamente extremo no constituye necesariamente un dato incorrecto. Su eliminación requeriría contrastar los precios con antecedentes comerciales, catálogos de productos o registros transaccionales.

Adicionalmente, el límite inferior calculado es negativo ($-250.010), por lo que el método IQR no identifica precios positivos extremadamente bajos. Esto explica por qué valores unitarios de $1 y $50 requieren una revisión complementaria.

En consecuencia, el conjunto final cumple con la detección de anomalías solicitada, pero sus indicadores monetarios deben interpretarse considerando la influencia de los valores atípicos.

## Informe comparativo de calidad

La columna **antes** utiliza los CSV unidos, antes de deduplicar. La columna **después** se calcula desde el DataFrame final. Los valores no se fijan manualmente en cero.

Para las columnas creadas durante el proceso y para el IQR se indica **No aplica** en la medición inicial. El cuadro de filas permite reconciliar: originales − duplicados − rechazadas = finales.

In [0]:
filas_despues = df_ventas_final.count()
duplicados_despues = filas_despues - df_ventas_final.dropDuplicates().count()
nulos_despues = contar_faltantes(df_ventas_final, df_ventas_final.columns)
cantidades_invalidas_despues = df_ventas_final.filter(F.col("cantidad").isNull()).count()
cantidad_no_positiva_despues = df_ventas_final.filter(F.col("cantidad") <= 0).count()
negativos_despues = df_ventas_final.filter(F.col("precio_total") < 0).count()

tabla_calidad = spark.createDataFrame([
    ("Filas totales", str(filas_antes), str(filas_despues)),
    ("Duplicados completos", str(duplicados_antes), str(duplicados_despues)),
    ("Categorías vacías/nulas", str(nulos_antes["categoria"]), str(nulos_despues["categoria"])),
    ("Cantidad inválida o nula", str(cantidad_invalida_antes + nulos_antes["cantidad"]), str(cantidades_invalidas_despues)),
    ("Cantidad menor o igual a cero", str(cantidad_no_positiva_antes), str(cantidad_no_positiva_despues)),
    ("Fecha inválida o nula", str(fechas_invalidas_antes), str(nulos_despues["fecha"])),
    ("Precio inválido o nulo", str(precios_invalidos_antes), str(nulos_despues["precio_unitario"])),
    ("Cantidad x precio negativa", str(totales_negativos_antes), str(negativos_despues)),
    ("Outliers de precio unitario", "No aplica", str(df_outliers.count()))
], ["indicador", "antes", "despues"])
tabla_calidad.show(truncate=False)

tabla_nulos = spark.createDataFrame([
    (columna, str(nulos_antes[columna]) if columna in nulos_antes else "No aplica", str(nulos_despues[columna]))
    for columna in df_ventas_final.columns
], ["columna", "faltantes_antes", "faltantes_despues"])
tabla_nulos.show(truncate=False)

print(f"Reconciliación: {filas_antes} - {duplicados_antes} - {filas_rechazadas} = {filas_despues}")
assert filas_antes - duplicados_antes - filas_rechazadas == filas_despues

+-----------------------------+---------+-------+
|indicador                    |antes    |despues|
+-----------------------------+---------+-------+
|Filas totales                |315      |274    |
|Duplicados completos         |15       |0      |
|Categorías vacías/nulas      |46       |0      |
|Cantidad inválida o nula     |0        |0      |
|Cantidad menor o igual a cero|27       |0      |
|Fecha inválida o nula        |0        |0      |
|Precio inválido o nulo       |0        |0      |
|Cantidad x precio negativa   |22       |0      |
|Outliers de precio unitario  |No aplica|7      |
+-----------------------------+---------+-------+

+-----------------+---------------+-----------------+
|columna          |faltantes_antes|faltantes_despues|
+-----------------+---------------+-----------------+
|id_transaccion   |0              |0                |
|fecha            |0              |0                |
|id_producto      |0              |0                |
|producto_nombre  |0     

# PARTE 5: Análisis con Spark SQL

Se crea la vista temporal ventas_limpias a partir del DataFrame final. Las tres consultas corresponden a las solicitadas en la actividad.

In [0]:
df_ventas_final.createOrReplaceTempView("ventas_limpias")
print("Vista temporal creada: ventas_limpias")

Vista temporal creada: ventas_limpias


## Consulta 1: diez ventas con mayor precio total

Esta consulta muestra el identificador, fecha, sucursal, producto y total de las ventas de mayor monto.

In [0]:
consulta_1 = spark.sql('''
    SELECT id_transaccion, fecha, sucursal, producto_nombre, precio_total
    FROM ventas_limpias
    ORDER BY precio_total DESC, id_transaccion ASC
    LIMIT 10
''')
consulta_1.show(truncate=False)

+--------------+----------+------------+---------------------+------------+
|id_transaccion|fecha     |sucursal    |producto_nombre      |precio_total|
+--------------+----------+------------+---------------------+------------+
|TRX20047      |2024-12-07|Santiago    |router wifi          |7.9999992E7 |
|TRX30045      |2024-10-01|Concepción  |laptop gamer         |7.999992E7  |
|TRX20001      |2024-11-23|Santiago    |disco ssd 1tb        |6.999993E7  |
|TRX20119      |2024-12-19|Santiago    |auriculares bluetooth|6.999993E7  |
|TRX10037      |2024-11-10|Viña del Mar|webcam hd            |3.5555552E7 |
|TRX20037      |2024-12-16|Santiago    |mouse inalambrico    |2.999997E7  |
|TRX30009      |2024-10-20|Concepción  |impresora láser      |1.999998E7  |
|TRX10074      |2024-11-25|Viña del Mar|teclado mecanico     |4499900.0   |
|TRX20014      |2024-10-15|Santiago    |teclado mecánico     |4499900.0   |
|TRX20063      |2024-10-17|Santiago    |laptop gamer         |4049910.0   |
+-----------

## Interpretación: transacciones de mayor monto

La consulta permite identificar las diez ventas de mayor valor monetario, ordenadas de manera descendente según `precio_total`.

La transacción de mayor monto corresponde a `TRX20047`, registrada en Santiago, asociada al producto `router wifi`, con un total de **$79.999.992**. Le sigue la transacción `TRX30045`, correspondiente a un `laptop gamer` vendido en Concepción, por **$79.999.920**.

Un hallazgo relevante es que las siete transacciones de mayor monto corresponden a registros previamente identificados como atípicos en sus precios unitarios.

Esto evidencia que las transacciones más elevadas no necesariamente representan las ventas comerciales más relevantes, sino que podrían estar influenciadas por anomalías en los precios registrados.

Por tanto, antes de utilizar estas operaciones para identificar productos de alto rendimiento o establecer estrategias comerciales, se recomienda verificar la validez de los precios unitarios asociados.

## Consulta 2: total vendido y transacciones por sucursal

Permite comparar el aporte de cada sucursal considerando el monto total y el número de transacciones válidas.

In [0]:
consulta_2 = spark.sql('''
    SELECT
        sucursal,
        SUM(precio_total) AS total_ventas,
        COUNT(*) AS cantidad_transacciones
    FROM ventas_limpias
    GROUP BY sucursal
    ORDER BY total_ventas DESC
''')
consulta_2.show(truncate=False)

+------------+------------+----------------------+
|sucursal    |total_ventas|cantidad_transacciones|
+------------+------------+----------------------+
|Santiago    |3.58320486E8|112                   |
|Concepción  |1.5611041E8 |70                    |
|Viña del Mar|1.24594447E8|92                    |
+------------+------------+----------------------+



## Interpretación: desempeño comercial por sucursal

La agregación de ventas permite comparar los montos acumulados y el número de registros válidos de cada sucursal.

**Santiago** presenta el mayor monto acumulado, con $358.320.486 distribuidos en 112 registros, seguido de **Concepción**, con $156.110.410 en 70 registros, y **Viña del Mar**, con $124.594.447 en 92 registros.

Santiago concentra aproximadamente el 56,07% del monto total registrado y el 40,88% de las observaciones válidas, por lo que lidera ambas métricas.

Sin embargo, los resultados deben interpretarse con cautela. La presencia de precios unitarios atípicos, especialmente en Santiago y Concepción, podría alterar considerablemente las diferencias monetarias observadas.

Asimismo, el conteo realizado mediante `COUNT(*)` corresponde al número de filas del conjunto de datos, por lo que su interpretación como cantidad de transacciones individuales supone que cada fila representa una transacción única.

En consecuencia, los resultados permiten describir el comportamiento de las ventas registradas, pero no son suficientes para concluir que una sucursal presenta mayor eficiencia comercial sin considerar la influencia de los valores atípicos y las características operacionales de cada establecimiento.

## Consulta 3: precios por categoría

Se calculan el promedio, mínimo y máximo del precio unitario para cada categoría.

In [0]:
consulta_3 = spark.sql('''
    SELECT
        categoria,
        AVG(precio_unitario) AS precio_promedio,
        MIN(precio_unitario) AS precio_minimo,
        MAX(precio_unitario) AS precio_maximo
    FROM ventas_limpias
    GROUP BY categoria
    ORDER BY categoria
''')
consulta_3.show(truncate=False)

+--------------+------------------+-------------+-------------+
|categoria     |precio_promedio   |precio_minimo|precio_maximo|
+--------------+------------------+-------------+-------------+
|Almacenamiento|200602.24489795917|25990.0      |449990.0     |
|Audio         |549928.4          |1.0          |9999990.0    |
|Computación   |144365.22916666666|1.0          |449990.0     |
|Periféricos   |177102.44444444444|50.0         |449990.0     |
|Redes         |392705.9761904762 |1.0          |9999990.0    |
|Sin categoría |1148463.95        |50.0         |9999990.0    |
+--------------+------------------+-------------+-------------+



## Interpretación: distribución de precios por categoría

El análisis de precios unitarios por categoría permite identificar diferencias en los valores promedio, mínimos y máximos de los productos comercializados.

La agrupación `Sin categoría` presenta el mayor precio promedio, alcanzando **$1.148.463,95**. Sin embargo, esta agrupación corresponde a registros cuya clasificación original estaba ausente, por lo que no debe interpretarse como una categoría comercial propiamente tal.

Entre las categorías identificadas, **Audio** presenta el mayor precio promedio ($549.928,40), seguida de **Redes** ($392.705,98). No obstante, ambas incluyen valores máximos de $9.999.990, lo que sugiere que sus promedios podrían estar fuertemente influenciados por observaciones extremas.

También se identifican precios mínimos de $1 y $50 en distintas categorías. Aunque estos valores no incumplen la regla de precios negativos, podrían corresponder a errores de registro, promociones excepcionales u otras situaciones que requieren verificación.

Por tanto, los resultados permiten comparar descriptivamente los precios registrados, pero el promedio aritmético debe interpretarse con precaución ante la presencia de valores extremos.


## Conclusiones generales del análisis

El desarrollo del taller permitió implementar un flujo completo de procesamiento de datos utilizando **Apache Spark en Databricks**, abarcando la integración, limpieza, transformación, validación y análisis de registros de ventas correspondientes a las sucursales de Santiago, Viña del Mar y Concepción.

En la etapa de integración, se consolidaron correctamente **315 registros**, verificando que el total coincidiera con la suma de los tres archivos originales. Posteriormente, el proceso de depuración permitió identificar y eliminar 15 registros duplicados, gestionar 46 categorías faltantes mediante una clasificación explícita y excluir 26 registros con cantidades no válidas, obteniendo un conjunto final de **274 registros**, equivalente al 86,98% de los datos originales.

Las transformaciones realizadas permitieron estandarizar los formatos de fechas, nombres de productos y precios unitarios, además de incorporar variables calculadas para facilitar el análisis. Por su parte, las reglas de validación confirmaron la ausencia de precios totales negativos y permitieron identificar **7 registros con precios unitarios atípicos** mediante el método del rango intercuartílico (IQR). Estos registros se conservaron debidamente identificados, considerando que un valor extremo no necesariamente constituye un error.

Mediante las consultas desarrolladas en **Spark SQL**, se identificaron las transacciones de mayor monto, se compararon las ventas acumuladas entre sucursales y se caracterizó la distribución de precios por categoría. Los resultados muestran que Santiago concentra el mayor monto de ventas, mientras que las diferencias observadas entre categorías y sucursales deben interpretarse considerando la influencia de los valores atípicos y las limitaciones de calidad detectadas.

En conclusión, el procesamiento permitió transformar tres fuentes de datos con problemas de calidad en un conjunto consolidado, depurado y validado, adecuado para desarrollar los análisis descriptivos solicitados. Asimismo, el ejercicio demuestra la importancia de aplicar controles de calidad y documentar las decisiones de limpieza antes de utilizar la información como apoyo para la toma de decisiones comerciales.